# MiniGrid: Exploratory Q-Learning Extension

> **Status: Experimental / Incomplete**
>
> This notebook is an exploratory extension of the main Frozen Lake project. It adapts the
> Q-learning implementation to the [MiniGrid](https://minigrid.farama.org/) environment,
> which uses image-based (partial) observations instead of a flat integer state.
>
> **This notebook is not runnable without installing the `minigrid` package:**
> ```
> pip install minigrid
> ```
>
> Even with the package installed, the tabular approach used here is a rough approximation:
> MiniGrid's observation space is too large for a standard Q-table. A proper treatment
> would require function approximation (e.g., DQN). This notebook documents the initial
> exploration and is preserved for research continuity.

---

## Key Difference from Frozen Lake

MiniGrid returns a **dictionary observation** containing a 7×7×3 partial view of the grid,
the agent's direction, and a mission string. Because Q-learning requires a discrete state
index, we flatten the image channel and the direction into a tuple and maintain a
dictionary (`bag`) that maps unique tuples to integer indices.

## 1. Setup

Run the cell below to check whether `minigrid` is available. If it raises an `ImportError`,
install the package with `pip install minigrid` and restart the kernel.

In [ ]:
import numpy as np
import gymnasium as gym
import time
import warnings
from collections import defaultdict

warnings.filterwarnings("ignore")

try:
    import minigrid  # registers MiniGrid environments with gymnasium
    print(f"minigrid {minigrid.__version__} available.")
    MINIGRID_AVAILABLE = True
except ImportError:
    print("minigrid is NOT installed.")
    print("Install with: pip install minigrid")
    print("Cells below will not run until the package is available.")
    MINIGRID_AVAILABLE = False

rng = np.random.default_rng(42)

## 2. Observation Preprocessing

MiniGrid observations are dictionaries. We extract the object-type channel of the image
(`obs['image'][:, :, 0]`) and the agent direction, then flatten them into a tuple that
can be used as a dictionary key.

In [ ]:
def preprocess(obs):
    """Map a MiniGrid dict observation to a hashable tuple.

    Extracts the object-type channel of the partial-view image and concatenates
    it with the agent's direction. The resulting tuple is used as a Q-table key
    via the `bag` dictionary.
    """
    image_flat = list(obs["image"][:, :, 0].flatten())
    return tuple(image_flat + [obs["direction"]])

## 3. Q-Learning (Tabular, Dictionary-Based)

Because the observation space is not a fixed-size integer, we maintain a dictionary
`bag` that assigns a running integer index to each distinct observation tuple seen
during training. The Q-table is pre-allocated with `nS` rows; if more unique states
are encountered than `nS`, the index will exceed the table bounds.

> **Known limitation:** This approach only works if the number of unique observations
> encountered during training stays below `nS`. For the small `Empty-5x5` environment
> this is usually satisfied, but it is not guaranteed and is not a scalable approach.

In [ ]:
def q_learning_minigrid(
    env,
    nS=200,
    nA=3,
    gamma=0.9,
    alpha=0.5,
    epsilon_start=1.0,
    epsilon_min=0.05,
    decay=0.99,
    epochs=500,
    max_steps=50,
    seed=42,
):
    """Tabular Q-learning for MiniGrid using a dictionary state index.

    Args:
        env    : MiniGrid Gymnasium environment
        nS     : Pre-allocated Q-table rows (upper bound on unique observations)
        nA     : Number of actions (3 for MiniGrid: turn-left, turn-right, forward)
        gamma  : Discount factor
        alpha  : Learning rate
        epsilon_start: Initial exploration rate
        epsilon_min  : Exploration rate floor
        decay  : Per-episode epsilon decay
        epochs : Number of training episodes
        max_steps: Maximum steps per episode
        seed   : RNG seed

    Returns:
        Q       : Q-value table, shape (nS, nA)
        bag     : Dict mapping observation tuple → Q-table row index
        rewards : Per-episode total rewards
    """
    rng_q = np.random.default_rng(seed)
    bag = {}   # observation tuple → integer state index
    idx = 0    # next free row in Q

    Q = np.zeros([nS, nA])
    epsilon = epsilon_start
    rewards = []

    for _ in range(epochs):
        obs, _ = env.reset()
        key = preprocess(obs)
        if key not in bag:
            bag[key] = idx
            idx += 1
        s = bag[key]

        epsilon = max(epsilon * decay, epsilon_min)
        ep_reward = 0.0

        for _ in range(max_steps):
            if rng_q.random() > epsilon:
                a = int(np.argmax(Q[s]))
            else:
                a = int(rng_q.integers(nA))

            obs, r, terminated, truncated, _ = env.step(a)
            key_next = preprocess(obs)
            if key_next not in bag:
                bag[key_next] = idx
                idx += 1
            s_next = bag[key_next]

            # Guard against table overflow
            if s_next < nS:
                Q[s, a] += alpha * (r + gamma * np.max(Q[s_next]) - Q[s, a])

            ep_reward += r
            s = s_next
            if terminated or truncated:
                break

        rewards.append(ep_reward)

    return Q, bag, rewards

## 4. Run Experiment

The cell below trains Q-learning on `MiniGrid-Empty-5x5-v0`.
It will fail if `minigrid` is not installed.

In [ ]:
if not MINIGRID_AVAILABLE:
    print("Skipping: minigrid not installed.")
else:
    import matplotlib.pyplot as plt

    env = gym.make("MiniGrid-Empty-5x5-v0")
    print(f"Environment: {env.spec.id}")
    print(f"Observation space: {env.observation_space}")
    print(f"Action space: {env.action_space}")

    Q, bag, rewards = q_learning_minigrid(
        env,
        nS=500,
        nA=3,     # MiniGrid Empty-5x5 uses: 0=turn-left, 1=turn-right, 2=forward
        gamma=0.9,
        alpha=0.5,
        epsilon_start=1.0,
        decay=0.99,
        epochs=500,
        max_steps=50,
        seed=42,
    )

    print(f"\nTraining complete. Unique observations seen: {len(bag)}")
    print(f"Mean reward (last 50 episodes): {np.mean(rewards[-50:]):.4f}")

    # Smoothed reward curve
    window = 50
    smoothed = np.convolve(rewards, np.ones(window) / window, mode="valid")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(rewards, alpha=0.3, color="steelblue", linewidth=0.8, label="episode reward")
    ax.plot(range(window - 1, len(rewards)), smoothed,
            color="steelblue", linewidth=2, label=f"{window}-ep moving avg")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title("MiniGrid-Empty-5x5 — Q-Learning Reward Curve", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    env.close()

## 5. Known Issues and Limitations

| Issue | Details |
|---|---|
| **Not runnable without `minigrid`** | Install with `pip install minigrid` |
| **Tabular Q-table is a rough fit** | MiniGrid observations are high-dimensional; the dictionary-index trick works for tiny maps only |
| **Q-table overflow risk** | If unique observations exceed `nS`, indices silently exceed table bounds |
| **No proper evaluation** | No systematic success-rate evaluation was added |
| **`render_single` not adapted** | Rendering requires `env.step` to return an integer obs, which MiniGrid does not |

## 6. Suggested Extensions

- Use **DQN** (deep Q-network) with a CNN to handle the image observations properly
- Use **Stable-Baselines3** for a production-quality baseline comparison
- Extend to more complex MiniGrid tasks (e.g., `MiniGrid-FourRooms-v0`) once a proper
  function approximator is in place